# AI Power Portfolio — Colab Quickstart

Clones the GitHub repo, installs dependencies, mounts Google Drive so the
SQLite warehouse persists between sessions, then runs DB init + ETL.

**Before running:** replace `REPO_URL` below with your actual GitHub repo URL.

Every cell below checks that the previous step actually succeeded before
continuing, and prints a plain-language explanation (not just a raw error)
if something's wrong — so if a cell stops with a message, read that message
first before asking for help.

In [ ]:
REPO_URL = "https://github.com/<your-username>/ai-power-portfolio.git"

# REPO_DIR is worked out automatically from REPO_URL above, using whatever
# your repo is actually named (e.g. "Capstone", "ai-power-portfolio", etc).
# You only need to edit REPO_URL — never edit this line by hand.
REPO_DIR = REPO_URL.rstrip("/").split("/")[-1]
if REPO_DIR.endswith(".git"):
    REPO_DIR = REPO_DIR[:-4]

if "<your-username>" in REPO_URL:
    raise ValueError(
        "REPO_URL still has the placeholder '<your-username>' in it.\n"
        "Go to your repository on GitHub, click the green 'Code' button, copy the "
        "HTTPS link shown there, and paste it in place of the URL above. Then re-run this cell."
    )

print(f"Repository URL: {REPO_URL}")
print(f"Will be cloned into folder: {REPO_DIR}")

## 1. Mount Google Drive (keeps the warehouse.db between sessions)

Colab wipes local VM files when the session ends. Mounting Drive and pointing
the warehouse there means your data survives across runs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA_DIR = "/content/drive/MyDrive/ai_power_portfolio_data"
import os
os.makedirs(DRIVE_DATA_DIR, exist_ok=True)

## 2. Clone the repo and install dependencies

This cell removes any previous (possibly failed) clone first, so it's always
safe to just re-run it rather than restarting the whole session. If the clone
fails, it explains the most common causes instead of just showing a raw git error.

In [ ]:
import subprocess, shutil, os

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

result = subprocess.run(["git", "clone", REPO_URL], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    err = result.stderr.lower()
    if "could not read username" in err or "authentication failed" in err or "terminal prompts disabled" in err:
        raise RuntimeError(
            "Clone failed because Git tried to ask for a username/password — this almost always "
            "means your repository is set to Private.\n"
            "Fix: on GitHub, go to your repo -> Settings -> scroll to the red 'Danger Zone' section "
            "-> Change visibility -> Make public. Then re-run this cell."
        )
    elif "not found" in err or "repository not found" in err:
        raise RuntimeError(
            "GitHub says this repository doesn't exist. Double-check REPO_URL in the first cell for typos — "
            "it should be an exact copy of the HTTPS link from the green 'Code' button on your repo page."
        )
    else:
        raise RuntimeError(f"git clone failed for an unexpected reason — see the output above.")

if not os.path.isdir(os.path.join(REPO_DIR, "src")):
    raise RuntimeError(
        f"Cloned '{REPO_DIR}' successfully, but there's no 'src' folder inside it.\n"
        "This means the repo on GitHub is missing files — go back and make sure you uploaded "
        "the ENTIRE contents of the ai_power_portfolio folder (including the src/ folder), not just some files."
    )

os.chdir(REPO_DIR)
print(f"\nSuccess — now working inside: {os.getcwd()}")

In [ ]:
!pip install -q -r requirements.txt

## 3. Point the warehouse at Drive instead of the ephemeral VM disk

In [ ]:
import os
os.environ["WAREHOUSE_DB_PATH"] = f"{DRIVE_DATA_DIR}/warehouse.db"

# settings.py reads this env var via python-dotenv/os.getenv at import time,
# so setting it before any `from src...` import is what matters here.

## 4. Initialize the warehouse and run the ETL

Each step's exit status is checked explicitly. If either fails, the cell stops
immediately with the real error instead of letting a later cell fail confusingly
with something like `no such table: fact_market_price`.

In [ ]:
import subprocess, sys

def run_module(module_name):
    print(f"--- running: python -m {module_name} ---")
    result = subprocess.run([sys.executable, "-m", module_name], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(
            f"'{module_name}' failed (see the error above). Common causes: the repo clone step above "
            "didn't actually succeed, or a required package failed to install. Scroll up and check "
            "those cells' output before re-running this one."
        )
    print(f"--- {module_name} finished OK ---\n")

run_module("src.db.init_db")
run_module("src.etl.load_market_data")

## 5. Sanity check — query the warehouse directly

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(os.environ["WAREHOUSE_DB_PATH"])
try:
    df = pd.read_sql("""
        SELECT a.ticker, d.full_date, f.close
        FROM fact_market_price f
        JOIN dim_asset a ON a.asset_key = f.asset_key
        JOIN dim_date d ON d.date_key = f.date_key
        ORDER BY d.full_date DESC
        LIMIT 20
    """, conn)
except Exception as e:
    print(f"Could not query the warehouse: {e}")
    print("This means Section 4 above didn't actually finish successfully — scroll up to that "
          "cell's output and re-run from there rather than re-running just this cell.")
    raise
finally:
    conn.close()
df

## 6. Run the test suite (optional)

In [ ]:
!python -m pytest tests/ -v